In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("AmazonReviews").master("local[*]").config("spark.sql.caseSensitive", "true").getOrCreate()
df_beauty = spark.read.json("All_Beauty.jsonl")

df_beauty.columns

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/01 17:44:02 WARN Utils: Your hostname, Pangkaews-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.63.13.171 instead (on interface en0)
26/03/01 17:44:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/01 17:44:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

['asin',
 'helpful_vote',
 'images',
 'parent_asin',
 'rating',
 'text',
 'timestamp',
 'title',
 'user_id',
 'verified_purchase']

In [2]:
from pyspark.sql.functions import col, when, size
from pyspark.sql.types import DoubleType

df_beauty = df_beauty.drop("images","asin")
df_beauty.columns

['helpful_vote',
 'parent_asin',
 'rating',
 'text',
 'timestamp',
 'title',
 'user_id',
 'verified_purchase']

In [3]:
# cleaning pipeline

from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StringType
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover
import re
import nltk
from nltk.stem import WordNetLemmatizer
import emoji
from pyspark.sql import functions as F

# download NLTK
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

# function to convert emoji to their meaning
def emoji_to_text(text):
    if not text:
        return ""
    return emoji.demojize(text, delimiters=(" ", " "))

# change 2 or more repeated character to 1 repeated character
def normalize_elongated(word):
    if not word:
        return ""
    return re.sub(r'(.)\1{2,}', r'\1\1', word)
    
# lemmatisation
def lemmatize_words(words):
    if not words:
        return []
    return [lemmatizer.lemmatize(w) for w in words]

# convert python function into UDF, so spark can use it
normalize_elongated_udf = F.udf(lambda words: [normalize_elongated(w) for w in words], ArrayType(StringType()))
emoji_to_text_udf = F.udf(emoji_to_text, StringType())
lemmatize_udf = F.udf(lemmatize_words, ArrayType(StringType()))


def preprocessing(df, text_col):
    # convert data to string
    df_cleaned = df.withColumn(text_col, F.coalesce(col(text_col).cast("string"), F.lit("")))
    
    # remove html and links
    df_cleaned = df_cleaned.withColumn(text_col, F.regexp_replace(col(text_col), r"http\S+|www\S+|<.*?>", " "))

    # convert emoji to their meaning
    df_cleaned = df_cleaned.withColumn(text_col, emoji_to_text_udf(col(text_col)))

    # lower case
    df_cleaned = df_cleaned.withColumn(text_col, F.lower(col(text_col)))

    # tokenisation
    tokenizer = RegexTokenizer(inputCol=text_col, outputCol=f"{text_col}_words", pattern="\\W")
    df_cleaned = tokenizer.transform(df_cleaned)

    # elongated word like soooo good -> so good
    df_cleaned = df_cleaned.withColumn(f"{text_col}_words", normalize_elongated_udf(col(f"{text_col}_words")))

    # remove stop word
    stopwords_remover = StopWordsRemover(inputCol=f"{text_col}_words", outputCol=f"{text_col}_no_stop")
    df_cleaned = stopwords_remover.transform(df_cleaned)
    
    #lemmatisation
    df_cleaned = df_cleaned.withColumn(f"{text_col}_lemma", lemmatize_udf(col(f"{text_col}_no_stop")))
    
    # join back to strings
    df_cleaned = df_cleaned.withColumn(f"{text_col}_final", F.concat_ws(" ", col(f"{text_col}_lemma")))
    
    # drop unneccesary columns
    df_cleaned = df_cleaned.drop(text_col, f"{text_col}_lemma", f"{text_col}_words", f"{text_col}_no_stop")
    df_cleaned = df_cleaned.withColumnRenamed(f"{text_col}_final", text_col)
    
    return df_cleaned

df_cleaned_title = preprocessing(df_beauty, text_col="title")
df_cleaned_text = preprocessing(df_cleaned_title, text_col="text")
#verify cleaning pipeline
df_cleaned_text .select("title","text").show(5, truncate=False)




[nltk_data] Downloading package wordnet to /Users/prim/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/prim/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
26/03/01 17:44:06 WARN StopWordsRemover: Default locale set was [en_TH]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.
26/03/01 17:44:06 WARN StopWordsRemover: Default locale set was [en_TH]; however, it was not found in available locales in JVM, falling back to en_US locale. Set param `locale` in order to respect another locale.
[Stage 1:>                                                          (0 + 1) / 1]

+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|title                        |text                                                                                                                                                                              |
+-----------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|lovely scent overpowering    |spray really nice smell really good go really fine trick say feel like need lot though get texture want lot hair medium thickness comparing brand yucky chemical m gonna stick try|
|work great smell little weird|product need wish odorless soft coconut smell head smell like orange coffee offputting granted know smell described hoping li

In [4]:
#handle date time data

from pyspark.sql import functions as F

# our time data is in milliseconds, so we have to convert into seconds for further use
df_with_date = df_cleaned_text.withColumn(
    "timestamp_convert", 
    (F.col("timestamp") / 1000).cast("timestamp")
)

# extract Year and Month for task3
df_with_date = df_with_date.withColumn("year", F.year("timestamp_convert"))
df_with_date = df_with_date.withColumn("month", F.month("timestamp_convert"))

# verify result
df_with_date.select("timestamp", "timestamp_convert", "year", "month").show(5)


# fill missing value in helpful_vite with 0
df_helpful = df_with_date.withColumn("helpful_vote", F.coalesce(F.col("helpful_vote"), F.lit(0)))

# convert rating to double for task1
df_rating = df_helpful.withColumn("rating_num", F.coalesce(col("rating").cast("double"), F.lit("")))

# convert verified_purchase column from Boolean to 1,0
df_ver_purchase = df_rating.withColumn("verified_purchase_num", F.when(F.col("verified_purchase") == True, 1).otherwise(0))

# verify converting
df_ver_purchase.select("verified_purchase_num","verified_purchase").show(10)
df_ver_purchase.printSchema()

# drop existing column to prepare to change column's name of new columns
df_drop = df_ver_purchase.drop("verified_purchase","rating")

# rename columns
df_final = df_drop.withColumnRenamed("verified_purchase_num", "verified_purchase")
df_final = df_final.withColumnRenamed("rating_num", "rating")

# verify result
print(f"final column : {df_final.columns}")
df_final.printSchema()

# save file
df_with_date.coalesce(1).write.mode("overwrite").json("cleaned_beauty")
print("save completed")

+-------------+--------------------+----+-----+
|    timestamp|   timestamp_convert|year|month|
+-------------+--------------------+----+-----+
|1588687728923|2020-05-05 15:08:...|2020|    5|
|1588615855070|2020-05-04 19:10:...|2020|    5|
|1589665266052|2020-05-16 22:41:...|2020|    5|
|1643393630220|2022-01-28 18:13:...|2022|    1|
|1609322563534|2020-12-30 10:02:...|2020|   12|
+-------------+--------------------+----+-----+
only showing top 5 rows
+---------------------+-----------------+
|verified_purchase_num|verified_purchase|
+---------------------+-----------------+
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    1|             true|
|                    0|            false|
|                    0|            false

[Stage 4:>                                                          (0 + 1) / 1]

save completed
